# Qwen anchor long retention compare (Colab)

Этот ноутбук запускает long-generation compare для **математических** и **кодовых** retention-подсказок:
- base `Qwen/Qwen2.5-1.5B`
- anchor-biased `Qwen/Qwen2.5-1.5B`

Сейчас по умолчанию включён `math` preset под интеграл с требованием использовать только integration by parts.


> Рекомендуемый runtime в Colab: **GPU**. На CPU 500 токенов может идти слишком долго.


In [ ]:
!nvidia-smi || true
!python --version
!pip install -q torch transformers einops pytest


In [ ]:
%cd /content
import os
if not os.path.exists('/content/ABPT'):
    !git clone https://github.com/kharkilirov1/Anchor-engine.git ABPT
%cd /content/ABPT


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'
DEVICE = 'cuda'
MAX_NEW_TOKENS = 500
MAX_LENGTH = 1024
SEED = 7
CONFLICT_THRESHOLD = 0.55
BIAS_SCALE = 1.50
REPETITION_PENALTY = 1.15
FREQUENCY_PENALTY = 0.05
NO_REPEAT_NGRAM_SIZE = 3
MIN_BIAS_PRESSURE = 0.60
MAX_BIAS_GATE_SUM = 1.50

ADAPTIVE_BIAS = True  # динамически масштабирует bias по уровню дрейфа
ADAPTIVE_BIAS_FLOOR = 0.05  # минимальный scale при adaptive режиме

PRESETS = {
    'code': {
        'prompt': 'You are a senior FastAPI engineer. Write a long implementation guide for an async FastAPI service with typed Pydantic models, dependency injection, async request handlers, validation, background jobs, and deployment notes. Do not rewrite the design as Django or synchronous class-based views.',
        'positive_keywords': 'fastapi,async,pydantic,await,dependency injection,request handler,typed request models',
        'negative_keywords': 'django,synchronous,template,class-based view,middleware stack,orm admin',
        'output_json': 'archive/colab_qwen_code_long_compare.json',
        'output_md': 'docs/research/colab_qwen_code_long_compare.md',
    },
    'math': {
        'prompt': 'Solve the integral ∫x²eˣdx using integration by parts only. Do not use tables or shortcuts.',
        'positive_keywords': 'integration by parts,let u,dv,du,v,repeat,first,second,therefore',
        'negative_keywords': 'table,tables,shortcut,shortcuts,formula sheet,lookup,substitution,computer algebra',
        'output_json': 'archive/colab_qwen_math_long_compare.json',
        'output_md': 'docs/research/colab_qwen_math_long_compare.md',
    },
}

PRESET = 'math'
PROMPT = PRESETS[PRESET]['prompt']
POSITIVE_KEYWORDS = PRESETS[PRESET]['positive_keywords']
NEGATIVE_KEYWORDS = PRESETS[PRESET]['negative_keywords']
OUTPUT_JSON = PRESETS[PRESET]['output_json']
OUTPUT_MD = PRESETS[PRESET]['output_md']


In [ ]:
import subprocess
import sys
from pathlib import Path

cmd = [
    sys.executable,
    'scripts/run_qwen_long_retention_compare.py',
    '--model', MODEL_NAME,
    '--device', DEVICE,
    '--prompt', PROMPT,
    '--max_new_tokens', str(MAX_NEW_TOKENS),
    '--max_length', str(MAX_LENGTH),
    '--seed', str(SEED),
    '--conflict_threshold', str(CONFLICT_THRESHOLD),
    '--bias_scale', str(BIAS_SCALE),
    '--repetition_penalty', str(REPETITION_PENALTY),
    '--frequency_penalty', str(FREQUENCY_PENALTY),
    '--no_repeat_ngram_size', str(NO_REPEAT_NGRAM_SIZE),
    '--min_bias_pressure', str(MIN_BIAS_PRESSURE),
    '--max_bias_gate_sum', str(MAX_BIAS_GATE_SUM),
    *(['--adaptive_bias'] if ADAPTIVE_BIAS else []),
    '--adaptive_bias_floor', str(ADAPTIVE_BIAS_FLOOR),
    '--positive_keywords', POSITIVE_KEYWORDS,
    '--negative_keywords', NEGATIVE_KEYWORDS,
    '--output_json', OUTPUT_JSON,
    '--output_md', OUTPUT_MD,
]
print('RUNNING:', ' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'compare script failed with code {result.returncode}')
assert Path(OUTPUT_JSON).exists(), f'missing output json: {OUTPUT_JSON}'
assert Path(OUTPUT_MD).exists(), f'missing output md: {OUTPUT_MD}'


In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))
print('base lexical score:', payload['base_analysis']['lexical_score'])
print('anchor lexical score:', payload['anchor_analysis']['lexical_score'])
print('base positive hits:', payload['base_analysis']['positive_hits'])
print('anchor positive hits:', payload['anchor_analysis']['positive_hits'])
print('base negative hits:', payload['base_analysis']['negative_hits'])
print('anchor negative hits:', payload['anchor_analysis']['negative_hits'])
print('anchor protected negative hits:', payload['anchor_analysis'].get('protected_negative_hits', {}))
print('anchor bias active steps:', sum(1 for step in payload['anchor']['steps'] if step.get('bias_nonzero_anchors', 0) > 0))
print('blocked ngram steps:', sum(1 for step in payload['anchor']['steps'] if step.get('blocked_ngram_token_count', 0) > 0))


In [ ]:
print('=== BASE CONTINUATION ===')
print(payload['base']['continuation_text'][:4000])
print()
print('=== ANCHOR CONTINUATION ===')
print(payload['anchor']['continuation_text'][:4000])


Сейчас notebook настроен на `PRESET = 'math'` и `MAX_NEW_TOKENS = 500`.
Если захочешь вернуться к коду, просто переключи `PRESET = 'code'`.
Guardrails против петли: repetition penalty, frequency penalty, no-repeat trigram, bias pressure gate, bias cap.
